In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.options.display.max_rows = 300
import datetime as dt
import math
import seaborn as sns
import miceforest as mf
from tableone import TableOne, load_dataset

# 根据定义筛选S-AKI患者

## sepsis 

In [ ]:
data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype_v3/00.data_eicu/disease_definition/sepsis/"
infile = data_dir + "eicu_sepsis3.csv"
df_sepsis = pd.read_csv(infile)

sepsis_lst = df_sepsis.stay_id.tolist()
print("sepsis患者数量为(by sepsis3)：",df_sepsis.shape[0])
df_sepsis.head(1)

## CKD

In [ ]:
data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype_v3/00.data_eicu/disease_definition//CKD/"
infile = data_dir + "eicu_ckd.csv"
df_ckd = pd.read_csv(infile)
ckd_lst = set(df_ckd.stay_id.tolist())
print("ckd患者数量为(by ICD)：",len(ckd_lst))

## AKI

In [ ]:
data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype_v3/00.data_eicu/disease_definition/AKI/"
infile = data_dir + "eicu_sk_first_and_max_stage.csv"
df_aki = pd.read_csv(infile)
df_aki = df_aki[["stay_id", "first_aki_onset"]]
df_aki.columns = ["stay_id", "aki_onset"]
aki_lst = df_aki.stay_id.tolist()
print("aki患者数量为(by KDIGO)：",len(aki_lst))


## sepsis without CKD;  sepsis with AKI

In [ ]:
sepsis_minus_ckd_lst = list(set(sepsis_lst).difference(ckd_lst))
print("sepsis_minus_ckd_lst: " +str(len(sepsis_minus_ckd_lst)))

sepsis_minusCKD_addAKI_lst = list(set(sepsis_minus_ckd_lst).intersection(aki_lst))

sepsis_noKidney_lst = list(set(sepsis_lst).difference(ckd_lst, aki_lst))
print("sepsis_minusCKD_addAKI_lst:",len(sepsis_minusCKD_addAKI_lst)," sepsis_noKidney:",len(sepsis_noKidney_lst))


# S-AKI 纳排标准

## 在sepsis后一周内发生AKI的病人

In [ ]:
df_sepsis = df_sepsis[df_sepsis["stay_id"].isin(sepsis_minusCKD_addAKI_lst)]
df_aki = df_aki[df_aki["stay_id"].isin(sepsis_minusCKD_addAKI_lst)]
df_sepsis_aki = pd.merge(df_sepsis, df_aki, how="inner", on = ["stay_id"])

df_sepsis_aki["time_aki_minus_sepsis"] =  df_sepsis_aki["aki_onset"] - df_sepsis_aki["sepsis_onset"]
df_sepsis_aki = df_sepsis_aki[(df_sepsis_aki["time_aki_minus_sepsis"]>0) & (df_sepsis_aki["time_aki_minus_sepsis"]<=168)]
saki_lst = df_sepsis_aki.stay_id.unique().tolist()
print("在sepsis后一周内发生AKI的病人数量：",len(saki_lst))


## 除去age<18, los<1

In [ ]:
df_demo =  pd.read_csv("/public/hanl/jupyter_dir/kidney_sepsis_penotype_v3/00.data_eicu/feature_data/df_eicu_sk_icudetails.csv")
df_demo = df_demo[df_demo["stay_id"].isin(saki_lst)]
df_demo = df_demo[df_demo["icu_los_hours"]>=24] # 条件1： ICU住院时长
df_demo = df_demo[(df_demo["age"]>=18)] # 条件2： age >= 18

saki_lst_filt_lst = list(df_demo.stay_id.unique())
print("saki, 年龄>17且los大于1天：",len(saki_lst_filt_lst))
all_lst = saki_lst_filt_lst

## 去除基线cre 大于1.5、小于0.5的患者

In [ ]:
df_base_crea = pd.read_csv("../00.data_eicu/disease_definition/AKI/df_base_crea.csv")
df_base_crea = df_base_crea[(df_base_crea["baseline_creatinine"]>=0.5) &(df_base_crea["baseline_creatinine"]<1.5)]
df_base_crea = df_base_crea[~df_base_crea.isnull().T.any()]

df_base_crea = df_base_crea[df_base_crea["stay_id"].isin(all_lst)]

all_lst = df_base_crea.stay_id.unique()
print("去掉没有基线 或基线过低 或基线已到肾衰竭或尿毒症的患者：",len(all_lst))

# 用于聚类的临床数据获得

## 导入文件

In [ ]:
# 导入时间文件
df_time =  pd.read_csv("/public/hanl/jupyter_dir/kidney_sepsis_penotype_v3/00.data_eicu/disease_definition/AKI/eicu_saki_event_time.csv")
df_time = df_time[(df_time["saki_onset"]<=df_time["outtime"]) & (df_time["saki_onset"]>=df_time["intime"])]
df_time = df_time[["stay_id","saki_onset"]]
df_time = df_time[df_time["stay_id"].isin(all_lst)]
print(len(df_time.stay_id.unique()))

In [ ]:
# 导入临床特征文件
data_dir = "/public/hanl/jupyter_dir/kidney_sepsis_penotype_v3/00.data_eicu/feature_data/"
infile = data_dir + "eicu_data_merge.csv"
df_fea = pd.read_csv(infile)
df_fea = df_fea[df_fea["stay_id"].isin(all_lst)]
df_fea["charttime"] = df_fea["charttime"]/60
df_fea = df_fea.drop_duplicates()
df_fea.head(1)

## 按时间窗口获取数据


In [ ]:
df_fea_add  = pd.merge(df_fea,df_time, how="left", on="stay_id")
df_fea_add["time"] = (df_fea_add["charttime"] - df_fea_add["saki_onset"])/6
df_fea_add["time"] = df_fea_add["time"].map(lambda x:x//1) 
df_fea_add["time"] = df_fea_add["time"].map(lambda x: x+1 if x>=0 else x)
df_fea_add = df_fea_add[(df_fea_add["time"]>=-4) &(df_fea_add["time"]<=28)]

df_fea_add = df_fea_add.drop(["charttime","saki_onset"],axis=1)
df_fea_add = df_fea_add.groupby(["stay_id","time"]).agg("mean").reset_index()

# condition2: SAKI发生后必须有一天及以上的记录
tmp_df = df_fea_add[df_fea_add["time"]>=0][["stay_id","time"]]
tmp_df = tmp_df.groupby("stay_id").agg(['max','count']).reset_index()
tmp_df.columns = ['_'.join(col) for col in tmp_df.columns]
stay_lst = tmp_df[(tmp_df["time_max"]>=4) & (tmp_df["time_count"]>=4)]["stay_id_"].tolist()
df_fea_add = df_fea_add[df_fea_add["stay_id"].isin(stay_lst)]
all_lst = df_fea_add.stay_id.unique()
print("在saki发生后有一天记录以上的患者：", len(all_lst))

#  condition4: SAKI最少有4个数据记录点
df_stage = pd.read_csv("/public/hanl/jupyter_dir/kidney_sepsis_penotype_v3/00.data_eicu//disease_definition/AKI/kdigo_aki_stage.csv")
df_stage = df_stage[df_stage["stay_id"].isin(all_lst)]
df_stage =df_stage[df_stage["aki_stage"]>0]
df_stage = df_stage.sort_values(["stay_id","time"])
df_stage = df_stage.groupby("stay_id").agg({"aki_stage":"count"}).reset_index()
df_stage = df_stage[df_stage.aki_stage >=4]
all_lst = df_stage.stay_id.unique()
print("AKI最少有4个数据记录点", len(all_lst))
df_fea_add = df_fea_add[df_fea_add["stay_id"].isin(all_lst)]

all_lst = df_fea_add.stay_id.unique()
print("最后纳入患者数量：",len(all_lst))
df_fea_add.head(1)

In [ ]:
# 尿量为空值的比例如此之大，前向插补，导致尿量激增
df_fea_add[df_fea_add["urineoutput"].isnull()].shape,df_fea_add[~df_fea_add["urineoutput"].isnull()].shape

## 前向插补

In [ ]:
df_fea_add["urineoutput"] = df_fea_add["urineoutput"].fillna(0)
# 前向插补
tmp_df = pd.DataFrame()
df_fea_add = df_fea_add.sort_values(by=["stay_id","time"])
for i,j in df_fea_add.groupby(["stay_id"]):
    j = j.fillna(method="ffill")
    tmp_df = pd.concat([tmp_df, j], axis=0)
df_im = tmp_df

In [ ]:
# 根据na找
df = df_im
tmp_df = pd.DataFrame(df.isna().sum()/df.shape[0])
tmp_df = tmp_df.reset_index()

fea_lst = tmp_df[tmp_df[0]<=0.6]["index"].tolist()
fea_lst.extend(["mbp","sbp","dbp"])
df_im = df_im[fea_lst]
df_im.to_csv("./df_im_By_ffill.csv",index=False)
len(fea_lst), str(fea_lst)

## 多重插补

In [ ]:
# 多重插补
df_im = pd.read_csv("./df_im_By_ffill.csv")

kernel = mf.ImputationKernel(
    df_im,
    datasets=3,
    save_all_iterations=True,
    random_state=10
)
kernel.mice(iterations = 3, n_jobs=-1)
df_im2 = kernel.complete_data(dataset=1)
df_im2.to_csv("df_im_By_MI.csv",index=False)
df_im2.head(1)

## 添加额外特征-肌酐基线肌酐变化率

In [ ]:
# add 肌酐变化数 作为特征
df_base_crea = pd.read_csv("/public/hanl/jupyter_dir/kidney_sepsis_penotype_v3/00.data_eicu/disease_definition/AKI/df_base_crea.csv")
df_feaf = pd.merge(df_im2,df_base_crea,how="inner",on="stay_id")
df_feaf["crea_divide_basecrea"] = round(df_feaf["creatinine"]/df_feaf["baseline_creatinine"],2)
df_feaf = df_feaf.drop(["baseline_creatinine"],axis=1)
df_feaf.to_csv("sk_feature_timescale_eicu.csv",index=False)
df_feaf.head(2)

## 输出
- 仅包含平均值

In [ ]:
df_feaf.to_csv("sk_feature_timescale_eicu.csv",index=False)
len(df_feaf.stay_id.unique())

In [ ]:
# 最后51912位患者聚类的起点确定： 最后决定从SAKI发作前的12h开始
df = df_feaf
tmp_df = df[df["time"].isin([-4,-3,-2,-1])]
tmp_df["time"].value_counts()/len(df.stay_id.unique())

In [ ]:
df = df_feaf[df_feaf["time"]>=-2]
df.to_csv("sk_feature_timescale_Fb2_eicu.csv",index=False)